# Bareq · 2. Train

One notebook for all four runs. Set `RUN` below: `dark`, `noisy`, `old` or `all`.

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**.
**Inputs:** the `bareq-data` dataset. To resume a run that stopped, also add this notebook's previous output.

Output: `/kaggle/working/bareq-run-<RUN>/` with the checkpoint, the ONNX file, the metrics and the log.

In [ ]:
RUN = "all"      # dark | noisy | old | all
SMOKE = True     # True: a few-minute test with a forced stop and resume. False: the real run.

REPO = "https://github.com/yzdvai/bareq.git"
REF = "main"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!rm -rf /tmp/bareq && git clone -q {REPO} /tmp/bareq && cd /tmp/bareq && git checkout -q {REF} && git log -1 --oneline
!pip install -q onnx onnxruntime 2>&1 | tail -n 1

In [ ]:
if SMOKE:
    # stop at step 200 as if the session had ended, then run again: it must resume and finish.
    # The two halves use different memory layouts, to see which is faster on this GPU.
    args = "--total-steps 400 --val-every 200 --val-limit 6 --eval-limit 6 --out /tmp/smoke-run"
    !rm -rf /tmp/smoke-run
    !cd /tmp/bareq && python -m bareq.train --run {RUN} {args} --stop-after 200 --nchw
    !cd /tmp/bareq && python -m bareq.train --run {RUN} {args}
else:
    !cd /tmp/bareq && BAREQ_COMMIT=$(git rev-parse HEAD) python -m bareq.train --run {RUN}